In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5" # Old model, supporting prefill

In [16]:
# Helper functions
def add_user_message(messages: list[dict], text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list[dict], text: str):
    messages.append({"role": "assistant", "content": text})    


def chat(messages: list[dict], 
         system_prompt:str|None=None, 
         temperature:float=1.0,
         stop_sequences:list[str]|None=None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system_prompt:
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        params["thinking"] = {"type": "disabled"} # Thinking is not supported when using assistant message prefill

    response = client.messages.create(**params)
    # print(response)
    return response.content[0].text   

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python|json|regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages =[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [55]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that parses an AWS ARN string and returns a dictionary with its components (partition, service, region, account-id, and resource). For example, 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket' should return the parsed components.",
  'format': 'python'},
 {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'company-logs'. The policy should include permissions for s3:GetObject and s3:ListBucket actions.",
  'format': 'json'},
 {'task': "Write a regular expression that validates AWS Access Key IDs, which start with 'AKIA' or 'ASIA' followed by exactly 16 alphanumeric characters (uppercase letters and numbers only).",
  'format': 'regex'}]

In [56]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [67]:
from pydantic import BaseModel

class TestUseCase(BaseModel):
    task: str
    format: str

In [58]:
def run_prompt(test_case: TestUseCase) -> str:
    """Merges the prompt with the test case input, then returns the result"""
    prompt = f"""
    Please solve the following test:

    {test_case.task}

    * Respond with only Python, JSON or a plain Regex
    * Do not add any comments or commentary or explanations
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    return chat(messages, stop_sequences=["```"])
    

In [44]:
def grade_by_model(test_case: TestUseCase, output: str) -> dict:
    """Grades the response using the model. Returns a grade of "good" or "bad"."""
    prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution

    Original task:
    <task>
    {test_case.task}
    </task>

    Solution to evaluate:
    <solution>
    {output}
    </solution>

    # Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific format:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your evaluation
    - "score": A number between 1-10

    Respond only with JSON. Keep your response concise and direct.
    Examples response shape:
    {{
        "strengths": [],
        "weaknesses": [],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [59]:
grade_by_model(
    TestUseCase(task="Write a Python function that takes a list of numbers and returns the sum of the even numbers.", format="python"), 
    "```python\ndef sum_even_numbers(numbers):\n    return sum(num for num in numbers if num % 2 == 0)\n```")

{'strengths': ['Concise and Pythonic implementation using generator expression',
  'Correct logic that properly filters even numbers using modulo operator',
  'Efficient single-pass solution with O(n) time complexity'],
 'weaknesses': ['Missing input validation - no handling for None, empty list, or non-numeric values',
  'No documentation (docstring) explaining function purpose, parameters, or return value',
  'No type hints for parameters and return value'],
 'reasoning': 'The solution correctly solves the core problem with clean, idiomatic Python code. However, it lacks production-ready features like input validation, error handling, and documentation that would be expected in AWS/enterprise environments where robustness is critical.',
 'score': 7}

In [63]:
import re
import ast 

def validate_json(text:str) -> int:
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text:str) -> int:
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text:str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response: str, test_case: TestUseCase) -> int:
    format = test_case.format
    if format == "python":
        return validate_python(response)
    elif format == "json":
        return validate_json(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError("Unknown task type")

In [69]:
grade_syntax(
    "def sum_even_numbers(numbers):\n    return sum(num for num in numbers if num % 2 == 0)", 
    TestUseCase(task="Write a Python function that takes a list of numbers and returns the sum of the even numbers.", format="python"))

10

In [60]:
def run_test_case(test_case: TestUseCase):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score: float = model_grade["score"]
    reasoning: str = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)    

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "score": score,
        "test_case": test_case.model_dump(),
        "reasoning": reasoning,
    }

In [61]:
def run_eval(dataset: list):
    """Loads the dataset and calls run_test_case for each one"""
    results = []
    for test_case in dataset:
        result = run_test_case(TestUseCase(**test_case))
        results.append(result)
    
    return results

In [70]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)


results = run_eval(dataset)

In [51]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution: Extract S3 Bucket Name from ARN\n\n```python\ndef extract_bucket_name(arn):\n    \"\"\"\n    Extracts the bucket name from an S3 bucket ARN.\n    \n    Args:\n        arn (str): S3 bucket ARN in format 'arn:aws:s3:::bucket-name' \n                   or 'arn:aws:s3:::bucket-name/key'\n    \n    Returns:\n        str: The bucket name extracted from the ARN\n    \n    Examples:\n        >>> extract_bucket_name('arn:aws:s3:::my-bucket')\n        'my-bucket'\n        >>> extract_bucket_name('arn:aws:s3:::my-bucket/path/to/key')\n        'my-bucket'\n    \"\"\"\n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # The bucket name (and potentially key) is after the third colon\n    # Format: arn:aws:s3:::bucket-name or arn:aws:s3:::bucket-name/key\n    if len(parts) >= 6:\n        bucket_and_key = parts[5]\n        \n        # If there's a key (indicated by '/'), extract just the bucket name\n        if '/' in bucket_and_key:\n            buc

In [71]:

from statistics import mean

average_score = mean(result["score"] for result in results)
print(f"Average score: {average_score}")

Average score: 9.166666666666666
